# Hiểu LoRA và các biến thể LoRA để fine-tune LLM

Notebook này tóm tắt nội dung bài viết về **LoRA-derived techniques** bằng tiếng Việt, theo hướng dễ học và có ví dụ nhỏ chạy được bằng `numpy`, `pandas`, `matplotlib`.

> Mục tiêu: sau notebook này, bạn có thể giải thích được LoRA, LoRA-FA, VeRA, Delta-LoRA, LoRA+, LoRA-drop khác nhau ở đâu và nên dùng ý tưởng nào trong tình huống nào.

## Cách học gợi ý

1. Đọc phần giải thích ngắn trước mỗi cell code.
2. Chạy từng cell từ trên xuống dưới.
3. Thay đổi các biến như `d_in`, `d_out`, `rank`, `n_layers` để thấy số tham số thay đổi ra sao.

## 1. Vấn đề: vì sao cần LoRA?

Fine-tuning truyền thống cập nhật toàn bộ trọng số của mô hình. Với LLM có hàng tỷ tham số, cách này tốn:

- bộ nhớ GPU để lưu trọng số, gradient, optimizer states;
- thời gian huấn luyện;
- chi phí lưu nhiều bản model đã fine-tune cho nhiều người dùng/tác vụ.

**LoRA** giải quyết bằng cách giữ nguyên trọng số gốc `W` và chỉ học một cập nhật hạng thấp:

$$W_{adapted} = W + \Delta W, \qquad \Delta W = B A$$

Trong đó:

- `W` là trọng số gốc, bị đóng băng;
- `A` chiếu từ chiều lớn xuống rank nhỏ `r`;
- `B` chiếu từ rank nhỏ quay lại chiều đầu ra;
- chỉ `A` và `B` được train.

Nếu `r` nhỏ hơn rất nhiều so với kích thước của `W`, số tham số trainable giảm mạnh.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("default")
np.random.seed(42)

## 2. Đếm tham số: full fine-tuning vs LoRA

Giả sử một linear layer có trọng số `W` kích thước `d_out × d_in`.

- Full fine-tuning train: `d_out * d_in` tham số.
- LoRA train: `A` có kích thước `r × d_in`, `B` có kích thước `d_out × r`, tổng là `r * (d_in + d_out)`.

In [ ]:
def full_params(d_in, d_out):
    return d_in * d_out


def lora_params(d_in, d_out, rank):
    return rank * (d_in + d_out)


def pct(part, whole):
    return 100 * part / whole

# Ví dụ gần với transformer projection lớn
# Bạn có thể đổi các số này để tự kiểm chứng.
d_in = 4096
d_out = 4096
rank = 8

full = full_params(d_in, d_out)
lora = lora_params(d_in, d_out, rank)

pd.DataFrame({
    "method": ["Full fine-tuning", "LoRA"],
    "trainable_params": [full, lora],
    "percent_vs_full": [100, pct(lora, full)],
})

**Ý chính:** Với `d_in = d_out = 4096`, `rank = 8`, LoRA chỉ train khoảng `0.39%` tham số của một layer so với full fine-tuning.

In [ ]:
ranks = [1, 2, 4, 8, 16, 32, 64]
rank_df = pd.DataFrame({
    "rank": ranks,
    "lora_trainable_params": [lora_params(d_in, d_out, r) for r in ranks],
})
rank_df["percent_vs_full"] = rank_df["lora_trainable_params"].apply(lambda x: pct(x, full))
rank_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rank_df["rank"], rank_df["percent_vs_full"], marker="o")
ax.set_title("LoRA trainable parameters vs full fine-tuning")
ax.set_xlabel("Rank r")
ax.set_ylabel("% tham số so với full fine-tuning")
ax.grid(True, alpha=0.3)
plt.show()

## 3. LoRA hoạt động thế nào trong forward pass?

Với input `x`, linear layer gốc cho output:

$$y = x W^T$$

LoRA thêm nhánh nhỏ:

$$y = x W^T + \alpha \cdot x A^T B^T$$

Trong triển khai phổ biến, có hệ số scale như `alpha / rank`. Khi deploy, ta có thể gộp cập nhật vào `W`:

$$W_{merged} = W + \alpha \cdot BA$$

Vì vậy LoRA gần như không thêm latency inference sau khi merge.

In [ ]:
# Toy demo: kiểm tra forward LoRA và merge W cho cùng kết quả.
batch = 3
small_d_in = 5
small_d_out = 4
small_rank = 2
alpha = 1.0

X = np.random.randn(batch, small_d_in)
W = np.random.randn(small_d_out, small_d_in)
A = np.random.randn(small_rank, small_d_in) * 0.01
B = np.random.randn(small_d_out, small_rank) * 0.01

# Forward chưa merge
y_lora = X @ W.T + alpha * (X @ A.T @ B.T)

# Merge cập nhật vào W rồi forward như linear bình thường
W_merged = W + alpha * (B @ A)
y_merged = X @ W_merged.T

print("Sai số lớn nhất giữa 2 cách:", np.max(np.abs(y_lora - y_merged)))

## 4. Bản đồ nhanh các biến thể

| Kỹ thuật | Cái gì được train? | Mục tiêu chính | Điểm cần nhớ |
|---|---|---|---|
| **LoRA** | Ma trận `A` và `B` riêng cho từng layer | Giảm số tham số fine-tune | Có thể merge vào `W` khi inference |
| **LoRA-FA** | Chỉ train `B`, đóng băng `A` | Giảm activation memory | `FA` = Frozen-A |
| **VeRA** | Vector scale nhỏ theo layer; `A`, `B` random và share | Giảm tham số hơn LoRA | Cực ít tham số trainable |
| **Delta-LoRA** | Train `A`, `B` và cập nhật gián tiếp `W` | Tăng chất lượng | Khó lưu nhiều adapter vì `W` không còn chung hoàn toàn |
| **LoRA+** | Vẫn train `A`, `B`, nhưng LR của `B` cao hơn | Tăng tốc và cải thiện nhẹ | Thay đổi optimizer/lr, không đổi kiến trúc lớn |
| **LoRA-drop** | Chỉ đặt LoRA ở layer quan trọng | Tăng tốc training | Chọn layer dựa trên độ lớn activation của nhánh LoRA |

## 5. So sánh tham số trainable giữa các biến thể

Cell dưới đây dùng công thức đơn giản để tạo trực giác. Giả sử mỗi layer có một projection kích thước `d_out × d_in`.

- LoRA: mỗi layer train `A` và `B`.
- LoRA-FA: chỉ train `B`, còn `A` đóng băng.
- VeRA: `A`, `B` random và share; mỗi layer chỉ train 2 vector scale cỡ `rank + d_out` trong ví dụ đơn giản này.
- LoRA-drop: chỉ dùng LoRA ở một phần các layer.

In [ ]:
def lora_fa_params(d_in, d_out, rank):
    # Chỉ B trainable: d_out x rank
    return d_out * rank


def vera_params_per_layer(d_in, d_out, rank):
    # Minh họa đơn giản: train vector d ở rank-space và b ở output-space.
    return rank + d_out


def lora_drop_params(d_in, d_out, rank, n_layers, keep_ratio):
    kept_layers = int(round(n_layers * keep_ratio))
    return kept_layers * lora_params(d_in, d_out, rank)

n_layers = 32
keep_ratio = 0.35

comparison = pd.DataFrame({
    "method": ["Full fine-tuning", "LoRA", "LoRA-FA", "VeRA", "LoRA-drop (35% layers)"],
    "trainable_params": [
        n_layers * full_params(d_in, d_out),
        n_layers * lora_params(d_in, d_out, rank),
        n_layers * lora_fa_params(d_in, d_out, rank),
        n_layers * vera_params_per_layer(d_in, d_out, rank),
        lora_drop_params(d_in, d_out, rank, n_layers, keep_ratio),
    ],
})
comparison["percent_vs_full"] = comparison["trainable_params"].apply(
    lambda x: pct(x, comparison.loc[0, "trainable_params"])
)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_df = comparison.copy()
ax.bar(plot_df["method"], plot_df["percent_vs_full"])
ax.set_title("Ước lượng % trainable params so với full fine-tuning")
ax.set_ylabel("% so với full fine-tuning")
ax.set_yscale("log")
ax.tick_params(axis="x", rotation=25)
ax.grid(True, axis="y", alpha=0.3)
plt.show()

## 6. LoRA-FA: Frozen-A

LoRA-FA giữ `A` cố định và chỉ train `B`.

Trực giác:

- `A` biến input thành biểu diễn rank thấp.
- Nếu không cần cập nhật `A`, ta giảm số tham số trainable và giảm phần bộ nhớ liên quan đến gradient/activation cho `A`.
- Đổi lại, không gian cập nhật bị hạn chế hơn LoRA chuẩn, nhưng thường vẫn đủ tốt trong nhiều bài toán.

In [ ]:
fa_demo = pd.DataFrame({
    "matrix_or_vector": ["A", "B", "W gốc"],
    "LoRA": ["train", "train", "freeze"],
    "LoRA-FA": ["freeze", "train", "freeze"],
})
fa_demo

## 7. VeRA: share ma trận random, chỉ học vector scale

VeRA còn tiết kiệm hơn LoRA:

- `A` và `B` là ma trận random, đóng băng, được share giữa nhiều layer.
- Mỗi layer chỉ học các vector scale nhỏ.

Trực giác dễ nhớ: thay vì học cả hai ma trận adapter cho từng layer, VeRA dùng một “bộ khung random chung” rồi học núm điều chỉnh nhỏ cho từng layer.

In [ ]:
vera_demo = pd.DataFrame({
    "component": ["A", "B", "vector d", "vector b", "W gốc"],
    "initialization": ["random", "random", "ones", "zeros", "pretrained"],
    "trainable": [False, False, True, True, False],
    "shared_across_layers": [True, True, False, False, False],
})
vera_demo

## 8. Delta-LoRA: chất lượng cao hơn nhưng khó vận hành như adapter rời

Delta-LoRA vẫn train `A`, `B`, nhưng còn dùng thay đổi của `BA` giữa các bước training để cập nhật gián tiếp `W`.

Điểm mạnh:

- Có thể cải thiện chất lượng vì `W` cũng được điều chỉnh.

Điểm yếu vận hành:

- Nếu `W` thay đổi theo từng fine-tuned model, nhà cung cấp không còn chỉ cần lưu một base model chung + adapter nhỏ.
- Điều này làm tăng chi phí lưu trữ/phục vụ nếu có nhiều khách hàng hoặc nhiều adapter.

Vì vậy, Delta-LoRA có thể hấp dẫn khi bạn tự kiểm soát một model riêng, nhưng kém hấp dẫn hơn nếu cần phục vụ rất nhiều fine-tuned variants.

In [ ]:
# Minh họa ý tưởng delta giữa hai bước training, không phải implementation đầy đủ.
A_t = np.random.randn(rank, d_in) * 0.01
B_t = np.random.randn(d_out, rank) * 0.01
A_next = A_t + np.random.randn(rank, d_in) * 0.001
B_next = B_t + np.random.randn(d_out, rank) * 0.001

low_rank_update_t = B_t @ A_t
low_rank_update_next = B_next @ A_next
delta_low_rank = low_rank_update_next - low_rank_update_t

print("Shape của delta cập nhật cho W:", delta_low_rank.shape)
print("Norm của delta:", np.linalg.norm(delta_low_rank))

## 9. LoRA+: cùng kiến trúc, khác learning rate

LoRA+ giữ nguyên ý tưởng LoRA nhưng dùng learning rate khác nhau cho `A` và `B`, thường là:

$$lr_B > lr_A$$

Trực giác:

- Trong LoRA chuẩn, `B` thường được khởi tạo bằng 0, nên lúc đầu nhánh LoRA chưa tác động lên output.
- Cho `B` learning rate lớn hơn giúp nhánh LoRA “bắt nhịp” nhanh hơn.
- `A` đã random từ đầu, nên có thể dùng learning rate nhỏ hơn để tinh chỉnh ổn định.

In [ ]:
steps = np.arange(1, 31)
# Đường cong minh họa, không phải kết quả thực nghiệm.
loss_lora = 1.0 * np.exp(-0.08 * steps) + 0.18
loss_lora_plus = 1.0 * np.exp(-0.14 * steps) + 0.15

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, loss_lora, label="LoRA: lr_A = lr_B")
ax.plot(steps, loss_lora_plus, label="LoRA+: lr_B > lr_A")
ax.set_title("Minh họa LoRA+ hội tụ nhanh hơn")
ax.set_xlabel("Training step")
ax.set_ylabel("Loss minh họa")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 10. LoRA-drop: không nhất thiết layer nào cũng cần LoRA

Ý tưởng:

1. Gắn LoRA vào nhiều layer.
2. Train thử trên một mẫu dữ liệu nhỏ trong vài epoch.
3. Đo độ lớn activation của nhánh LoRA ở từng layer.
4. Giữ LoRA ở layer có đóng góp lớn, bỏ ở layer đóng góp nhỏ.
5. Train chính thức chỉ với các layer được giữ.

Lợi ích chính thường là giảm thời gian/chi phí training, không nhất thiết là tăng accuracy.

In [ ]:
# Demo chọn layer dựa trên score activation giả lập.
n_layers_demo = 12
activation_scores = np.abs(np.random.normal(loc=0.5, scale=0.25, size=n_layers_demo))
threshold = np.quantile(activation_scores, 0.60)  # giữ top 40%
keep = activation_scores >= threshold

layer_selection = pd.DataFrame({
    "layer": np.arange(n_layers_demo),
    "lora_activation_score": activation_scores.round(3),
    "keep_lora": keep,
})
layer_selection

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["tab:green" if k else "tab:gray" for k in keep]
ax.bar(layer_selection["layer"], layer_selection["lora_activation_score"], color=colors)
ax.axhline(threshold, color="red", linestyle="--", label="ngưỡng chọn")
ax.set_title("LoRA-drop: giữ adapter ở layer có score cao")
ax.set_xlabel("Layer")
ax.set_ylabel("Activation score minh họa")
ax.legend()
plt.show()

## 11. Chọn biến thể nào? Decision helper

Không có lựa chọn đúng cho mọi trường hợp. Bảng dưới đây là heuristic để ghi nhớ.

In [ ]:
def recommend_variant(priority):
    priority = priority.lower().strip()
    mapping = {
        "simple": "LoRA - baseline mạnh, dễ dùng, dễ merge.",
        "memory": "LoRA-FA - giảm memory bằng cách đóng băng A.",
        "few_params": "VeRA - cực ít tham số trainable nhờ share A/B và chỉ học vector scale.",
        "quality": "Delta-LoRA - cân nhắc nếu chấp nhận cập nhật W và không cần phục vụ nhiều adapter rời.",
        "speed": "LoRA+ hoặc LoRA-drop - LoRA+ đổi learning rate; LoRA-drop giảm số layer có adapter.",
        "many_customers": "LoRA/LoRA-FA/VeRA - ưu tiên adapter nhỏ, không sửa W gốc.",
    }
    return mapping.get(priority, "Hãy chọn priority: simple, memory, few_params, quality, speed, many_customers")

for p in ["simple", "memory", "few_params", "quality", "speed", "many_customers"]:
    print(f"{p:>14}: {recommend_variant(p)}")

## 12. Tóm tắt bằng một câu cho mỗi kỹ thuật

- **LoRA:** freeze model gốc, học cập nhật hạng thấp `BA`.
- **LoRA-FA:** giống LoRA nhưng freeze `A`, chỉ train `B` để tiết kiệm hơn.
- **VeRA:** dùng `A`, `B` random/share, chỉ học vector scale nhỏ.
- **Delta-LoRA:** dùng thay đổi của adapter để cập nhật thêm `W`, có thể tốt hơn nhưng khó lưu/phục vụ nhiều bản.
- **LoRA+:** cùng adapter LoRA, nhưng dùng learning rate cho `B` lớn hơn `A`.
- **LoRA-drop:** chọn layer quan trọng để đặt LoRA thay vì đặt ở mọi layer.

## Bài tập tự kiểm tra

1. Nếu tăng `rank` từ 8 lên 64 thì số tham số LoRA tăng bao nhiêu lần?
2. Vì sao LoRA có thể không thêm inference latency sau khi merge?
3. Vì sao Delta-LoRA không lý tưởng cho nhà cung cấp phải phục vụ hàng nghìn fine-tuned models?
4. Khi GPU memory rất hạn chế, bạn sẽ thử LoRA-FA hay LoRA+ trước? Vì sao?
5. Nếu cần giảm số tham số tối đa, VeRA có lợi thế gì so với LoRA?